# Notebook 03 — Front YOLOv8n Detection Baseline Training

**Objective:** reproduce the fixed Front detection baseline `FRONT_DET_YOLOV8N_001` with YOLOv8n, preserve the audited dataset byte-for-byte, and save compact reproducibility evidence. Training occurs only when the USER runs this notebook manually in VS Code.

This notebook does not modify labels, convert polygons, work on the Top camera, or execute Notebook 04.

## 1. Experiment metadata and runtime environment

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, subprocess, sys, time, warnings
import yaml
import torch
import ultralytics

EXPERIMENT_ID = 'FRONT_DET_YOLOV8N_001'
STARTED_AT = datetime.now(timezone.utc).isoformat()
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PATHS_CONFIG = PROJECT_ROOT / 'configs' / 'paths.yaml'
DATA_SOURCES_CONFIG = PROJECT_ROOT / 'configs' / 'data_sources.yaml'
GIT_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()
CONDA_ENV = os.environ.get('CONDA_DEFAULT_ENV', '')
CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'NOT_AVAILABLE'
print(f'experiment_id: {EXPERIMENT_ID}')
print(f'datetime_utc: {STARTED_AT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Conda environment: {CONDA_ENV}')
print(f'Python: {platform.python_version()}')
print(f'Torch: {torch.__version__}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'CUDA available: {CUDA_AVAILABLE}')
print(f'CUDA runtime: {torch.version.cuda}')
print(f'GPU: {GPU_NAME}')
print(f'Git commit: {GIT_COMMIT}')

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/diy-hus/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
experiment_id: FRONT_DET_YOLOV8N_001
datetime_utc: 2026-08-17T05:23:19.483512+00:00
PROJECT_ROOT: /home/diy-hus/fish
Python executable: /home/diy-hus/miniconda3/envs/fish/bin/python
Conda environment: fish
Python: 3.11.15
Torch: 2.13.0+cu130
Ultralytics: 8.4.120
CUDA available: True
CUDA runtime: 13.0
GPU: NVIDIA GeForce RTX 3050
Git commit: b3b6d2fadcdd5e38f434cc2041eac51ba773943e


## 2. CONFIG — USER confirmation gate

The old baseline epochs and batch size are not recorded in the repository. Set `EPOCHS` and `BATCH` deliberately before Run All. `BATCH` may be a fixed integer or `-1` for Ultralytics auto-batch. The notebook stops here while either value is unresolved.

In [3]:
MODEL = 'yolov8n.pt'
IMGSZ = 640
EPOCHS = 100  # USER_CONFIRM_REQUIRED: set an intentional baseline value
BATCH = 16   # USER_CONFIRM_REQUIRED: set an integer or -1 for auto-batch
DEVICE = 0
SEED = 42
WORKERS = 4
OPTIMIZER = 'auto'
RUNS_PROJECT = PROJECT_ROOT / 'runs' / 'front'
RUN_NAME = 'yolov8n_front_v1_baseline'
RUN_DIR_EXPECTED = RUNS_PROJECT / RUN_NAME
DATASET_ROOT = PROJECT_ROOT / 'data' / 'roboflow' / 'front_detect_v1'
DATA_YAML = DATASET_ROOT / 'data.yaml'
DATASET_MANIFEST = PROJECT_ROOT / 'results' / 'detection' / 'front_dataset_manifest.json'
EXPECTED_TRAIN_IMAGES, EXPECTED_TRAIN_INSTANCES = 1015, 3888
EXPECTED_VALID_IMAGES, EXPECTED_VALID_INSTANCES = 254, 995
USER_CONFIRM_REQUIRED = EPOCHS is None or BATCH is None
CONFIG = {
    'experiment_id': EXPERIMENT_ID, 'model': MODEL, 'data': str(DATA_YAML.relative_to(PROJECT_ROOT)),
    'imgsz': IMGSZ, 'epochs': EPOCHS, 'batch': BATCH, 'device': DEVICE, 'seed': SEED,
    'workers': WORKERS, 'optimizer': OPTIMIZER, 'project': str(RUNS_PROJECT.relative_to(PROJECT_ROOT)), 'name': RUN_NAME,
}
print('CONFIG')
print(yaml.safe_dump(CONFIG, sort_keys=False))
print(f'Model initialization/download target: {MODEL}')
print(f'USER_CONFIRM_REQUIRED: {USER_CONFIRM_REQUIRED}')
if USER_CONFIRM_REQUIRED:
    raise RuntimeError('USER_CONFIRM_REQUIRED: set EPOCHS and BATCH in the CONFIG cell, then Restart Kernel and Run All.')

CONFIG
experiment_id: FRONT_DET_YOLOV8N_001
model: yolov8n.pt
data: data/roboflow/front_detect_v1/data.yaml
imgsz: 640
epochs: 100
batch: 16
device: 0
seed: 42
workers: 4
optimizer: auto
project: runs/front
name: yolov8n_front_v1_baseline

Model initialization/download target: yolov8n.pt
USER_CONFIRM_REQUIRED: False


## 3. Preflight: configuration, dataset structure, and provenance

In [4]:
assert CONDA_ENV == 'fish', f'FAIL preflight: expected Conda env fish, found {CONDA_ENV!r}'
assert CUDA_AVAILABLE, 'FAIL preflight: CUDA is required because DEVICE=0.'
assert PATHS_CONFIG.is_file(), f'FAIL preflight: missing {PATHS_CONFIG}'
assert DATA_SOURCES_CONFIG.is_file(), f'FAIL preflight: missing {DATA_SOURCES_CONFIG}'
assert DATASET_ROOT.is_dir(), f'FAIL preflight: missing dataset root {DATASET_ROOT}'
assert DATA_YAML.is_file(), f'FAIL preflight: missing {DATA_YAML}'
assert DATASET_MANIFEST.is_file(), f'FAIL preflight: missing {DATASET_MANIFEST}'
if RUN_DIR_EXPECTED.exists() and any(RUN_DIR_EXPECTED.iterdir()):
    raise RuntimeError(f'FAIL preflight: run directory already contains files; preserve it and choose a new approved run name: {RUN_DIR_EXPECTED}')
with PATHS_CONFIG.open(encoding='utf-8') as handle:
    PATHS = yaml.safe_load(handle)
with DATA_SOURCES_CONFIG.open(encoding='utf-8') as handle:
    DATA_SOURCES = yaml.safe_load(handle)
with DATA_YAML.open(encoding='utf-8') as handle:
    DATA_DEFINITION = yaml.safe_load(handle)
with DATASET_MANIFEST.open(encoding='utf-8') as handle:
    DATASET_MANIFEST_CONTENT = json.load(handle)
FRONT_SOURCE = DATA_SOURCES['sources']['labeled_detection_dataset']['front']
assert FRONT_SOURCE['workspace'] == 'phys-hus'
assert FRONT_SOURCE['project'] == 'fish_front_detection'
assert int(FRONT_SOURCE['version']) == 1
assert FRONT_SOURCE['local_root'] == 'data/roboflow/front_detect_v1'
print(f'Dataset root: {DATASET_ROOT.relative_to(PROJECT_ROOT)}')
print(f'data.yaml: {DATA_YAML.relative_to(PROJECT_ROOT)}')
print(f'train path declaration: {DATA_DEFINITION.get("train")}')
print(f'valid path declaration: {DATA_DEFINITION.get("val")}')
print(f'class names: {DATA_DEFINITION.get("names")}')
print(f'dataset source: {FRONT_SOURCE}')
print(f'dataset manifest: {DATASET_MANIFEST.relative_to(PROJECT_ROOT)}')

Dataset root: data/roboflow/front_detect_v1
data.yaml: data/roboflow/front_detect_v1/data.yaml
train path declaration: ../train/images
valid path declaration: ../valid/images
class names: ['Ca']
dataset source: {'source': 'roboflow', 'workspace': 'phys-hus', 'project': 'fish_front_detection', 'version': 1, 'format': 'yolov8', 'local_root': 'data/roboflow/front_detect_v1'}
dataset manifest: results/detection/front_dataset_manifest.json


## 4. Preflight: load train and valid through Ultralytics detect loader

This uses the installed Ultralytics dataset API without rewriting annotations. Framework warnings remain visible and are also retained for the final evidence. A loader failure stops execution before model initialization or training.

In [5]:
import logging
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import LOGGER

class WarningCollector(logging.Handler):
    def __init__(self):
        super().__init__(level=logging.WARNING)
        self.messages = []
    def emit(self, record):
        self.messages.append(self.format(record))

warning_collector = WarningCollector()
LOGGER.addHandler(warning_collector)
PYTHON_WARNINGS = []
try:
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter('always')
        ULTRALYTICS_DATA = check_det_dataset(str(DATA_YAML), autodownload=False)
        TRAIN_DATASET = YOLODataset(img_path=ULTRALYTICS_DATA['train'], data=ULTRALYTICS_DATA, task='detect', imgsz=IMGSZ, augment=False, cache=False, prefix='preflight train: ')
        VALID_DATASET = YOLODataset(img_path=ULTRALYTICS_DATA['val'], data=ULTRALYTICS_DATA, task='detect', imgsz=IMGSZ, augment=False, cache=False, prefix='preflight valid: ')
        PYTHON_WARNINGS = [str(item.message) for item in captured]
finally:
    LOGGER.removeHandler(warning_collector)
ULTRALYTICS_WARNINGS = list(dict.fromkeys(warning_collector.messages + PYTHON_WARNINGS))
TRAIN_INSTANCES = sum(len(label['cls']) for label in TRAIN_DATASET.labels)
VALID_INSTANCES = sum(len(label['cls']) for label in VALID_DATASET.labels)
print(f'Ultralytics train images: {len(TRAIN_DATASET)}; instances: {TRAIN_INSTANCES}')
print(f'Ultralytics valid images: {len(VALID_DATASET)}; instances: {VALID_INSTANCES}')
print(f'Ultralytics warnings ({len(ULTRALYTICS_WARNINGS)}):')
for message in ULTRALYTICS_WARNINGS:
    print(f'- {message}')
assert len(TRAIN_DATASET) == EXPECTED_TRAIN_IMAGES, 'FAIL preflight: train image count differs from accepted audit.'
assert len(VALID_DATASET) == EXPECTED_VALID_IMAGES, 'FAIL preflight: valid image count differs from accepted audit.'
assert TRAIN_INSTANCES == EXPECTED_TRAIN_INSTANCES, 'FAIL preflight: train instance count differs from accepted audit.'
assert VALID_INSTANCES == EXPECTED_VALID_INSTANCES, 'FAIL preflight: valid instance count differs from accepted audit.'
print('PREFLIGHT_RESULT: PASS_WITH_WARNING' if ULTRALYTICS_WARNINGS else 'PREFLIGHT_RESULT: PASS')

preflight train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1737.0±1023.4 MB/s, size: 60.8 KB)
preflight train: Scanning /home/diy-hus/fish/data/roboflow/front_detect_v1/train/labels... 1015 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1015/1015 1.1Kit/s 0.9s1ss
preflight train: New cache created: /home/diy-hus/fish/data/roboflow/front_detect_v1/train/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 41, len(boxes) = 3888. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
preflight valid: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2536.0±607.0 MB/s, size: 54.8 KB)
preflight valid: Scanning /home/diy-hus/fish/data/roboflow/front_detect_v1/valid/labels... 254 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 254/254 1.2Kit/s 0.2s<0.0s
preflight valid: New cache created: /home/diy-hus/fish/data/roboflow/front_detect_

## 5. Train YOLOv8n baseline — USER execution only

Ultralytics output is not redirected. Epoch, GPU memory, losses, instances, and validation metrics remain visible in this cell. If `yolov8n.pt` is absent, Ultralytics may download the official pretrained weight when the USER runs this cell.

In [6]:
from ultralytics import YOLO
print(f'Initializing model: {MODEL}')
MODEL_OBJECT = YOLO(MODEL)
TRAIN_START = time.perf_counter()
TRAIN_RESULTS = MODEL_OBJECT.train(
    data=str(DATA_YAML), model=MODEL, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH,
    device=DEVICE, seed=SEED, workers=WORKERS, optimizer=OPTIMIZER, deterministic=True,
    project=str(RUNS_PROJECT), name=RUN_NAME, exist_ok=False, verbose=True, plots=True, val=True,
)
TRAINING_RUNTIME_SEC = time.perf_counter() - TRAIN_START
RUN_DIR = Path(MODEL_OBJECT.trainer.save_dir).resolve()
print(f'Training completed in {TRAINING_RUNTIME_SEC:.3f} s')
print(f'Run directory: {RUN_DIR}')

Initializing model: yolov8n.pt
Ultralytics 8.4.120 🚀 Python-3.11.15 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/diy-hus/fish/data/roboflow/front_detect_v1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, moment

## 6. Validate outputs, hash best model, and extract metrics

In [7]:
import pandas as pd
BEST_MODEL = RUN_DIR / 'weights' / 'best.pt'
LAST_MODEL = RUN_DIR / 'weights' / 'last.pt'
RESULTS_CSV = RUN_DIR / 'results.csv'
RESULTS_PNG = RUN_DIR / 'results.png'
assert BEST_MODEL.is_file(), f'FAIL: best.pt was not produced: {BEST_MODEL}'
assert RESULTS_CSV.is_file(), f'FAIL: results.csv was not produced: {RESULTS_CSV}'
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()
BEST_MODEL_SHA256 = sha256_file(BEST_MODEL)
METRICS_HISTORY = pd.read_csv(RESULTS_CSV)
METRICS_HISTORY.columns = [column.strip() for column in METRICS_HISTORY.columns]
required_metrics = ['epoch', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']
missing_metrics = [column for column in required_metrics if column not in METRICS_HISTORY.columns]
assert not missing_metrics, f'FAIL: required metrics missing from results.csv: {missing_metrics}'
best_index = METRICS_HISTORY['metrics/mAP50-95(B)'].astype(float).idxmax()
BEST_ROW = METRICS_HISTORY.loc[best_index]
BEST_EPOCH = int(BEST_ROW['epoch']) + 1
PRECISION = float(BEST_ROW['metrics/precision(B)'])
RECALL = float(BEST_ROW['metrics/recall(B)'])
MAP50 = float(BEST_ROW['metrics/mAP50(B)'])
MAP50_95 = float(BEST_ROW['metrics/mAP50-95(B)'])
print(f'best.pt: {BEST_MODEL.relative_to(PROJECT_ROOT)} ({BEST_MODEL.stat().st_size} bytes)')
print(f'last.pt exists: {LAST_MODEL.is_file()}')
print(f'results.csv: {RESULTS_CSV.relative_to(PROJECT_ROOT)} ({len(METRICS_HISTORY)} epochs)')
print(f'results.png exists: {RESULTS_PNG.is_file()}')
print(f'best.pt SHA-256: {BEST_MODEL_SHA256}')

best.pt: runs/front/yolov8n_front_v1_baseline/weights/best.pt (6242858 bytes)
last.pt exists: True
results.csv: runs/front/yolov8n_front_v1_baseline/results.csv (100 epochs)
results.png exists: True
best.pt SHA-256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738


## 7. Save compact evidence

In [8]:
LOG_DIR = PROJECT_ROOT / 'logs' / 'detection' / EXPERIMENT_ID
RESULTS_DIR = PROJECT_ROOT / 'results' / 'detection'
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = LOG_DIR / 'config.yaml'
ENVIRONMENT_PATH = LOG_DIR / 'environment.txt'
SUMMARY_PATH = LOG_DIR / 'summary.json'
METRICS_PATH = RESULTS_DIR / 'front_yolov8n_baseline_metrics.csv'
CONFIG_EVIDENCE = dict(CONFIG, dataset_source='roboflow', workspace='phys-hus', project_id='fish_front_detection', dataset_version=1, git_commit=GIT_COMMIT)
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG_EVIDENCE, sort_keys=False), encoding='utf-8')
ENVIRONMENT_LINES = [
    f'experiment_id={EXPERIMENT_ID}', f'datetime_utc={STARTED_AT}', f'git_commit={GIT_COMMIT}',
    f'python_executable={sys.executable}', f'python={platform.python_version()}', f'conda_env={CONDA_ENV}',
    f'torch={torch.__version__}', f'cuda_runtime={torch.version.cuda}', f'cuda_available={CUDA_AVAILABLE}',
    f'gpu={GPU_NAME}', f'ultralytics={ultralytics.__version__}',
]
ENVIRONMENT_PATH.write_text('\n'.join(ENVIRONMENT_LINES) + '\n', encoding='utf-8')
METRICS_EVIDENCE = pd.DataFrame([{
    'experiment_id': EXPERIMENT_ID, 'best_epoch': BEST_EPOCH, 'precision': PRECISION, 'recall': RECALL,
    'mAP50': MAP50, 'mAP50_95': MAP50_95, 'training_runtime_sec': round(TRAINING_RUNTIME_SEC, 3),
    'best_model_sha256': BEST_MODEL_SHA256, 'dataset_version': 1, 'git_commit': GIT_COMMIT,
}])
METRICS_EVIDENCE.to_csv(METRICS_PATH, index=False)
FINAL_WARNINGS = list(ULTRALYTICS_WARNINGS)
CHECKPOINT_RESULT = 'PASS_WITH_WARNING' if FINAL_WARNINGS else 'PASS'
SUMMARY = {
    'experiment_id': EXPERIMENT_ID, 'dataset': str(DATASET_ROOT.relative_to(PROJECT_ROOT)), 'dataset_version': 1,
    'model': MODEL, 'epochs': EPOCHS, 'batch': BATCH, 'best_epoch': BEST_EPOCH, 'precision': PRECISION,
    'recall': RECALL, 'mAP50': MAP50, 'mAP50_95': MAP50_95, 'training_runtime_sec': round(TRAINING_RUNTIME_SEC, 3),
    'best_model': str(BEST_MODEL.relative_to(PROJECT_ROOT)), 'best_model_sha256': BEST_MODEL_SHA256,
    'dataset_manifest': str(DATASET_MANIFEST.relative_to(PROJECT_ROOT)), 'dataset_source': FRONT_SOURCE,
    'git_commit': GIT_COMMIT, 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': FINAL_WARNINGS,
    'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in (CONFIG_PATH, ENVIRONMENT_PATH, SUMMARY_PATH, METRICS_PATH)],
    'next_step': 'USER reviews and accepts Notebook 03 results before any Notebook 04 work.',
}
SUMMARY_PATH.write_text(json.dumps(SUMMARY, indent=2, ensure_ascii=False), encoding='utf-8')
for path in (CONFIG_PATH, ENVIRONMENT_PATH, SUMMARY_PATH, METRICS_PATH):
    print(f'Created {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size} bytes)')

Created logs/detection/FRONT_DET_YOLOV8N_001/config.yaml (382 bytes)
Created logs/detection/FRONT_DET_YOLOV8N_001/environment.txt (333 bytes)
Created logs/detection/FRONT_DET_YOLOV8N_001/summary.json (1801 bytes)
Created results/detection/front_yolov8n_baseline_metrics.csv (296 bytes)


## 8. Final Summary

In [9]:
FINAL_SUMMARY = {
    'experiment_id': EXPERIMENT_ID, 'dataset': str(DATASET_ROOT.relative_to(PROJECT_ROOT)), 'dataset_version': 1,
    'model': MODEL, 'epochs': EPOCHS, 'best_epoch': BEST_EPOCH, 'precision': PRECISION, 'recall': RECALL,
    'mAP50': MAP50, 'mAP50_95': MAP50_95, 'training_runtime': f'{TRAINING_RUNTIME_SEC:.3f} sec',
    'best_model': str(BEST_MODEL.relative_to(PROJECT_ROOT)), 'best_model_sha256': BEST_MODEL_SHA256,
    'checkpoint_result': CHECKPOINT_RESULT, 'warnings': FINAL_WARNINGS,
    'output_files': SUMMARY['output_files'], 'next_step': SUMMARY['next_step'],
}
print('FINAL SUMMARY')
for key, value in FINAL_SUMMARY.items():
    print(f'{key}: {value}')

FINAL SUMMARY
experiment_id: FRONT_DET_YOLOV8N_001
dataset: data/roboflow/front_detect_v1
dataset_version: 1
model: yolov8n.pt
epochs: 100
best_epoch: 74
precision: 0.96682
recall: 0.9663
mAP50: 0.98976
mAP50_95: 0.55233
training_runtime: 1334.623 sec
best_model: runs/front/yolov8n_front_v1_baseline/weights/best.pt
best_model_sha256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738
checkpoint_result: PASS_WITH_WARNING
warnings: ['WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 41, len(boxes) = 3888. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.', 'WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 10, len(boxes) = 995. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.']
ou